# Phase 6 — Cybersecurity intelligence (version-aware correlation)

Correlates a local CVE snapshot against the real UTHP component versions. The key move is **version-aware** matching: a name match is only reported as *affected* if the version is below the CVE's fixed version. This is what suppresses the zlib false positive.

In [2]:
import sys; sys.path.append('../src')
import build_adapter, intel_adapter, ssc_graph as kg

build_adapter.generate('../data/uthp_image.rootfs.manifest', '../data/uthp_build.ttl')
out, n, stats = intel_adapter.correlate('../data/uthp_build.ttl', '../data/intel_snapshot.json', '../data/uthp_intel.ttl')
print(f'intel adapter -> {n} triples; {stats}')

intel adapter -> 57 triples; {'affected': 2, 'not_affected': 1}


In [3]:
g = kg.load(['../data/uthp_build.ttl', '../data/uthp_context.ttl', '../data/uthp_intel.ttl'])
print(len(g), 'triples after inference')

1137 triples after inference


In [4]:
print('NAIVE name+version matches (what a scanner would surface):')
for r in kg.q(g, '''SELECT ?comp ?cve ?vex WHERE {
    ?m ssc:matchesComponent ?comp ; ssc:matchesVulnerability ?cve ;
       ssc:hasVEXStatus/ssc:vexStatus ?vex . } ORDER BY ?comp'''):
    print('  ', ' | '.join(kg.short(x) for x in r))

print('\nGROUNDED (version-aware) AFFECTED only:')
for r in kg.q(g, '''SELECT ?comp ?cve WHERE {
    ?m ssc:matchesComponent ?comp ; ssc:matchesVulnerability ?cve ;
       ssc:hasVEXStatus/ssc:vexStatus ssc:Affected . } ORDER BY ?comp'''):
    print('  ', ' | '.join(kg.short(x) for x in r))

NAIVE name+version matches (what a scanner would surface):
   pkg_glibc_2_39 | CVE-2024-2961 | Affected
   pkg_openssh_9_6p1 | CVE-2024-6387 | Affected
   pkg_zlib_1_3_1 | CVE-2022-37434 | NotAffected

GROUNDED (version-aware) AFFECTED only:
   pkg_glibc_2_39 | CVE-2024-2961
   pkg_openssh_9_6p1 | CVE-2024-6387


## What just happened (the whole thesis in one screen)
- **openssh 9.6p1 → CVE-2024-6387 (regreSSHion)**: AFFECTED — RCE on the gateway, which `mayImpactFunction braking` via the CAN pivot.
- **glibc 2.39 → CVE-2024-2961**: AFFECTED.
- **zlib 1.3.1 → CVE-2022-37434**: name matches, but version 1.3.1 ≥ fixed 1.2.13 → **NotAffected**. The false positive is suppressed by grounding.

A naive scanner (or ungrounded LLM) flags all three; the grounded graph correctly keeps only the two real ones. That difference is exactly what the ablation (Phase 10) measures.

**Next (Phase 9):** feed a retrieved subgraph (affected component → CVE → CWE → impact path → evidence) to the local LLM so it *explains* the finding, grounded vs ungrounded.